In [1]:
import torch
from torch import nn
import torch.optim as optim

from torchvision import datasets, models, transforms

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import time

In [ ]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights
from itertools import chain

weights = ResNet18_Weights.DEFAULT

# Normalización estándar de ImageNet para ResNet18.
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]


# Transformaciones de entrenamiento para aumentar la variabilidad de las imágenes:
# - RandomResizedCrop: recorta y redimensiona aleatoriamente a 224x224
# - RandomHorizontalFlip: aplica volteo horizontal aleatorio
# - ToTensor: convierte la imagen a tensor
# - Normalize: normaliza con la media y desviación estándar de ImageNet
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

# Transformación de validación del modelo preentrenado.
test_transform = weights.transforms()

# Dataset de entrenamiento con imágenes de gatos y perros.
train_data = datasets.OxfordIIITPet(
    root="data",
    split="trainval",
    download=True,
    transform = train_transform
)

# Dataset de prueba para medir el rendimiento final.
test_data = datasets.OxfordIIITPet(
    root="data",
    split="test",
    download=True,
    transform = test_transform
)

# Cargamos ResNet18 con pesos preentrenados.
model = resnet18(weights=weights)
# Ajustamos la última capa a las 37 clases del dataset.
model.fc = nn.Linear(512, 37)
# Usamos GPU si está disponible.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Congelar toda la red
for parameter in model.parameters():
    parameter.requires_grad = False

# Solo entrenamos el último bloque convolucional.
for parameter in model.layer4.parameters():
    parameter.requires_grad = True
    
# Descongelar únicamente la última capa
for parameter in model.fc.parameters():
    parameter.requires_grad = True
    
# Función de pérdida para clasificación multiclase.
criterion = nn.CrossEntropyLoss()
# Optimizamos solo las capas que sí se entrenan.
optimizer = optim.Adam(chain(model.layer4.parameters(), model.fc.parameters()), lr = 1e-4)

# Batches para entrenar y evaluar el modelo.
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [ ]:
def train(model, train_loader, criterion, optimizer, epochs=10, device="cpu"):
    # Mide el tiempo total de entrenamiento.
    inicio = time.time()
    
    model.train() # Activa el modo entrenamiento (BatchNorm, Dropout, etc.)
    
    # Guarda la evolución del loss.
    loss_history = []
    # Guarda la evolución de accuracy.
    accuracy_history = []
    
    for epoch in range(epochs):
        correct = 0 # Cantidad de imágenes clasificadas correctamente.
        total = 0 # Cantidad total de imágenes procesadas.
        total_loss = 0 # Suma del loss de todos los batches de la época.
        
        for images, labels in train_loader:
            images = images.to(device)  # Mueve las imágenes al dispositivo seleccionado (CPU/GPU).
            labels = labels.to(device)  # Mueve las etiquetas al mismo dispositivo para poder compararlas con las predicciones.
            
            optimizer.zero_grad() # Reinicia los gradientes acumulados del paso anterior.
            logits = model(images) # Forward: obtiene los logits del modelo.
            loss = criterion(logits, labels) # Calcula la pérdida comparando logits vs etiquetas reales.
            loss.backward() # Backpropagation: Calcula automáticamente todos los gradientes.
            optimizer.step() # Actualiza pesos de las convoluciones, de las capas densas y bias utilizando esos gradientes.
            
            total_loss += loss.item() # Acumula el loss de este batch.
            predictions = logits.argmax(dim=1) # Elige el logit más grande de cada imagen. Esa será la clase predicha.
            correct += (predictions == labels).sum().item() # Cuenta cuántas imágenes fueron clasificadas correctamente.
            total += labels.size(0) # Acumula la cantidad total de imágenes vistas.

        loss_history.append(total_loss / len(train_loader)) # Loss promedio considerando todos los batches.
        accuracy_history.append(correct / total) # Porcentaje de aciertos sobre todo el dataset.

        print(
            f"Epoch {epoch+1}/{epochs} "
            f"- Loss: {loss_history[-1]:.4f} "
            f"- Accuracy: {accuracy_history[-1]*100:.2f}%"
        )
    
    # Mide el tiempo total de entrenamiento.
    fin = time.time()
    print(f"Tiempo: {fin-inicio:.2f} segundos")
    
    return loss_history, accuracy_history

def evaluate(model, eval_loader, device="cpu"):
    # Mide el tiempo de evaluación.
    inicio = time.time()
    
    model.eval() # Cambia el modelo a modo evaluación
    
    # Contadores para accuracy final.
    correct = 0
    total = 0
    
    with torch.no_grad(): # Desactiva el cálculo de gradientes para ahorrar memoria y computación.
        for images, labels in eval_loader:
            images = images.to(device)  # Mueve las imágenes al dispositivo seleccionado (CPU/GPU).
            labels = labels.to(device)  # Mueve las etiquetas al mismo dispositivo para poder compararlas con las predicciones.

            logits = model(images) # En evaluación solo hacemos forward pass; no actualizamos pesos.

            # La clase predicha vuelve a salir del logit de mayor valor.
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
    accuracy = correct / total # Accuracy global sobre todo el loader.
            
    # Tiempo total de evaluación.
    fin = time.time()
    
    print(f"Accuracy: {accuracy*100:.2f}%")   
    print(f"Tiempo: {fin-inicio:.2f} segundos")
          
    return accuracy

In [4]:
loss_history, accuracy_history = train(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=10,
    device=device
)

Epoch 1/10 - Loss: 2.0105 - Accuracy: 56.01%
Epoch 2/10 - Loss: 0.9194 - Accuracy: 79.89%
Epoch 3/10 - Loss: 0.6953 - Accuracy: 83.18%
Epoch 4/10 - Loss: 0.5678 - Accuracy: 86.06%
Epoch 5/10 - Loss: 0.5133 - Accuracy: 86.22%
Epoch 6/10 - Loss: 0.4581 - Accuracy: 88.15%
Epoch 7/10 - Loss: 0.4172 - Accuracy: 89.10%
Epoch 8/10 - Loss: 0.4042 - Accuracy: 88.80%
Epoch 9/10 - Loss: 0.3489 - Accuracy: 90.84%
Epoch 10/10 - Loss: 0.3201 - Accuracy: 91.33%
Tiempo: 239.59 segundos


In [5]:
acc = evaluate(
    model,
    test_loader,
    device=device
)

Accuracy: 89.56%
Tiempo: 29.99 segundos
